In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Define the Model class
class Model(nn.Module):
  def __init__(self, in_features = 432, h1 = 128, h2 = 64, out_features = 1):
    super().__init__()
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1, h2)
    self.out = nn.Linear(h2, out_features)
 # Initialize the layers of the model
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.out(x)
    return x

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
model = Model()

In [ ]:
# Load the dataset
# Replace 'your_dataset.csv' with the actual path to your dataset
data = pd.read_csv()
X = data.drop('target', axis = 1).values
y = data['target'].values

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scale Features
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to Tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [ ]:
# Training the model
# Set the number of epochs and initialize a list to store losses
epochs = 100
losses = []
for i in range(epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  losses.append(loss)

  if i % 5 == 0:
    print(f'Epoch: {i} Loss: {loss}')

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [ ]:
# Plotting the training loss
plt.plot(range(epochs), losses)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss over Time")
plt.show()

In [ ]:
# Evaluate the model
with torch.no_grad():
  y_eval = model(X_test)
  loss = criterion(y_eval, y_test)
  predicted = torch.argmax(y_eval, dim = 1)
  accuracy = (predicted == y_test).float().mean()
print(f'Test Loss: {loss.item():.4f}')
print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')